# FinCore Account Record Change Deep Agent Example

This notebook tests the Account Record Change module in two modes:

1. Deterministic domain flow with no LLM or API key.
2. LangChain Deep Agents flow with an Ollama-backed local LLM.

The LLM is intentionally above the domain services. It can coordinate tools and explain results, but policy decisions and command execution remain inside deterministic services.

## Setup

From the repository root, start Jupyter with the project environment:

```bash
uv run python -m ipykernel install --user --name fincore --display-name "FinCore"
uv run jupyter lab
```

The default LLM profile uses local Ollama with `qwen2.5:3b`. Make sure Ollama is running and the model is available:

```bash
ollama pull qwen2.5:3b
ollama serve
```

You can override the local model settings with:

```bash
export FINCORE_LLM_MODEL="qwen2.5:3b"
export FINCORE_LLM_BASE_URL="http://localhost:11434"
```

In [1]:
import os
from pprint import pprint

from fincore.account_record_change.deep_agent import (
    AccountRecordChangeDeepAgent,
    QWEN_OLLAMA_PROFILE,
    build_sample_private_name_update_request,
    create_chat_model,
    create_llm_deep_agent,
)
from fincore.account_record_change.models import ChangeRequest
from fincore.account_record_change.repositories import InMemoryRecordRepository
from fincore.account_record_change.deep_agent.llm_agent import _demo_record

MODEL_PROFILE = QWEN_OLLAMA_PROFILE.model_copy(update={
    "model": os.getenv("FINCORE_LLM_MODEL", "qwen2.5:3b"),
    "base_url": os.getenv("FINCORE_LLM_BASE_URL") or None,
})

## Build The Example Request

This uses the same schema shape as `docs/flow-example.md`.

In [2]:
request_payload = build_sample_private_name_update_request(new_name="Rahul K. Kumar")
pprint(request_payload)

{'account_type': 'PRIVATE_INDIVIDUAL',
 'changes': [{'action': 'REPLACE',
              'field_path': 'account_holder_name',
              'new_value': 'Rahul K. Kumar',
              'old_value': 'Rahul Kumar',
              'reason': 'Customer requested a name correction'}],
 'correlation_id': None,
 'entity_type': 'ACCOUNT',
 'evidence': [{'bounding_box': None,
               'document_id': 'DOC-99218',
               'document_type': 'IDENTITY_PROOF',
               'page': None,
               'purpose': 'NAME_CHANGE'}],
 'expected_record_version': 18,
 'idempotency_key': 'branch-portal-784512',
 'operation': 'UPDATE',
 'reason': None,
 'record_id': 'ACC-100582',
 'request_id': 'REQ-20260717-00124',
 'requested_by': {'authentication_level': 'MFA',
                  'branch_id': 'PUNE-017',
                  'channel': 'BRANCH_PORTAL',
                  'role': 'OPERATIONS_USER',
                  'user_id': 'USR-501'},
 'source_channel': None,
 'submitted_at': '2026-07-17T04:16:35

## Deterministic Preflight Without LLM

This verifies the domain module itself. It should pause for human review because the requested name is compatible but below the strict private-name threshold.

In [3]:
domain_agent = AccountRecordChangeDeepAgent(
    record_repository=InMemoryRecordRepository({"ACC-100582": _demo_record()})
)

request = ChangeRequest.model_validate(request_payload)
result = await domain_agent.run(request)

print("status:", result.state.status)
print("disposition:", result.state.decision.disposition if result.state.decision else None)
print("interrupt:", result.interrupt.interrupt_type if result.interrupt else None)
print("explanation:", result.explanation)
print("validation results:")
for validation in result.state.validation_results:
    pprint(validation.model_dump(mode="json"))

status: REVIEW_REQUIRED
disposition: HUMAN_REVIEW
interrupt: HUMAN_REVIEW
explanation: Official disposition is HUMAN_REVIEW, calculated by policy version 2026.07.1. Reason codes: PRIVATE_NAME_MATCH_BELOW_THRESHOLD.
validation results:
{'blocking': False,
 'confidence': None,
 'evidence_references': [],
 'expected_value': 18,
 'field_paths': ['record.version'],
 'message': 'Expected record version matches current record version.',
 'observed_value': 18,
 'reason_code': 'RECORD_VERSION_MATCH',
 'retryable': False,
 'rule_id': 'RECORD_VERSION_MATCH',
 'rule_version': '1.0',
 'severity': 'CRITICAL',
 'status': 'PASS',
 'tool_execution_id': None,
 'validation_execution_id': 'VAL-3ee09076-5eee-42b6-822d-bd98243a83ab',
 'validator_name': 'RECORD_VERSION_VALIDATOR'}
{'blocking': False,
 'confidence': None,
 'evidence_references': [],
 'expected_value': ['account_holder_name', 'customer_id', 'country'],
 'field_paths': ['account_holder_name', 'customer_id', 'country'],
 'message': 'Mandatory ac

## Actual LLM Deep Agent Invocation

This cell creates a real LangChain Deep Agent using `deepagents.create_deep_agent`. The model is created first from `MODEL_PROFILE`, then that model object is passed into the Deep Agent factory.

For this small local model, the recommended path is the one-shot sample tool:

- `run_sample_private_name_update_flow`

The lower-level tools are still available for full structured requests:

- `build_sample_private_name_update_request`
- `run_account_record_change_flow`
- `resume_account_record_change_review`

The LLM should preserve the exact requested new name and explain the official policy result rather than inventing a decision.

In [4]:
chat_model = create_chat_model(MODEL_PROFILE)
llm_agent = create_llm_deep_agent(model=chat_model)

llm_result = await llm_agent.ainvoke(
    {
        "messages": [
            {
                "role": "user",
                "content": (
                    "Use the one-shot sample tool run_sample_private_name_update_flow "
                    "with new_name exactly Rahul K. Kumar. Then explain the official result. "
                    "Do not build a partial request. Do not change the name to Rahul Kumar. "
                    "Do not invent policy decisions."
                ),
            }
        ]
    }
)
pprint(llm_result)

{'files': {},
 'messages': [HumanMessage(content='Use the one-shot sample tool run_sample_private_name_update_flow with new_name exactly Rahul K. Kumar. Then explain the official result. Do not build a partial request. Do not change the name to Rahul Kumar. Do not invent policy decisions.', additional_kwargs={}, response_metadata={}, id='15438607-a2b8-42f1-88b0-8e120f90f17a'),
              AIMessage(content='', additional_kwargs={}, response_metadata={'model': 'qwen2.5:3b', 'created_at': '2026-07-17T04:16:49.232924Z', 'done': True, 'done_reason': 'stop', 'total_duration': 12713693500, 'load_duration': 163139042, 'prompt_eval_count': 2050, 'prompt_eval_duration': 10771219000, 'eval_count': 35, 'eval_duration': 1670398000, 'logprobs': None, 'model_name': 'qwen2.5:3b', 'model_provider': 'ollama'}, id='lc_run--019f6e4a-6295-7f81-a6f9-15906fdc18b4-0', tool_calls=[{'name': 'run_sample_private_name_update_flow', 'args': {'new_name': 'Rahul K. Kumar', 'execute': True}, 'id': 'd4477a1e-461b-4f

## Resume After Human Review

If the LLM flow pauses for human review, the review can be resumed through the domain coordinator or through the LLM tool. This direct call is deterministic and does not require another LLM call.

In [5]:
if result.interrupt and result.interrupt.interrupt_type == "HUMAN_REVIEW":
    resumed = await domain_agent.resume_after_review(
        request_id=request.request_id,
        approval_reference="REV-NOTEBOOK-APPROVED",
        execute=False,
    )
    print("status:", resumed.state.status)
    pprint(resumed.output.model_dump(mode="json") if resumed.output else None)
    pprint(resumed.state.command.model_dump(mode="json") if resumed.state.command else None)
else:
    print("No human-review interrupt to resume.")

status: VALIDATED
{'disposition': 'HUMAN_REVIEW',
 'execution_result': None,
 'field_results': [{'decision': 'HUMAN_REVIEW',
                    'field_path': 'account_holder_name',
                    'validation_execution_ids': ['VAL-f9a295b6-6d95-4bc0-801f-604e2ecfc187',
                                                 'VAL-84877976-b51b-465b-9c19-225985f3228f']}],
 'policy_version': '2026.07.1',
 'request_id': 'REQ-20260717-00124',
 'review_task': {'review_task_id': 'REV-f626961a-bb9a-48a8-9808-11875a0afb04'},
 'status': 'VALIDATED'}
{'approval_reference': 'REV-NOTEBOOK-APPROVED',
 'command_id': 'CMD-REQ-20260717-00124-53a19158',
 'command_type': 'UPDATE_ACCOUNT_RECORD',
 'expected_record_version': 18,
 'mutations': [{'field_path': 'account_holder_name',
                'new_value': 'Rahul K. Kumar',
                'old_value': 'Rahul Kumar'}],
 'policy_decision_reference': 'DEC-2f065040-3c07-4fec-a578-d43b8773e572',
 'record_id': 'ACC-100582',
 'request_id': 'REQ-20260717-00124'}

## Auto-Approval Variant

Set the new value equal to the current authoritative name. This should auto-approve and prepare an update command.

In [6]:
auto_request_payload = build_sample_private_name_update_request(new_name="Rahul Kumar")
auto_request = ChangeRequest.model_validate(auto_request_payload)
auto_result = await domain_agent.run(auto_request)

print("status:", auto_result.state.status)
print("disposition:", auto_result.state.decision.disposition if auto_result.state.decision else None)
pprint(auto_result.state.command.model_dump(mode="json") if auto_result.state.command else None)

status: VALIDATED
disposition: AUTO_APPROVE
{'approval_reference': None,
 'command_id': 'CMD-REQ-20260717-00124-b71fdfdb',
 'command_type': 'UPDATE_ACCOUNT_RECORD',
 'expected_record_version': 18,
 'mutations': [{'field_path': 'account_holder_name',
                'new_value': 'Rahul Kumar',
                'old_value': 'Rahul Kumar'}],
 'policy_decision_reference': 'DEC-34b54d00-6bf0-4001-8dcc-e5d1ea276839',
 'record_id': 'ACC-100582',
 'request_id': 'REQ-20260717-00124'}
